# Import

In [1]:
import warnings
# warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="tqdm")

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import shutil

from torch.optim.adamax import Adamax


from torch.utils.data import random_split

from torchvision import transforms
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from utils.StegoDataset import StegoDataset
from utils.StegoPairDataSet import StegoPairDataset
from utils.StegoPairDataLoader import StegoPairDataLoader

from utils.checkpoint import save_checkpoint

from utils.transform import (
    RandomFlip,
    RandomRot,
    ToNumpy
)

from utils.transform import (
    RandomFlipList,
    RandomRotList,
    ToNumpyList,
    ToTensorList
)

from model.KeNet import KeNet as SiaStegNet
from model.losses import ContrastiveLoss

# Make DataSet

In [3]:
image_collection_folder_path = os.path.join(
    '..',
    '..',
    '..',
    'images',
    'BOSSbase',
)

cover_image_folder_path = os.path.join(image_collection_folder_path, 'cover')
stego_image_folder_path = os.path.join(image_collection_folder_path, 'stego')

transform = transforms.Compose([
    ToNumpyList(),
    # RandomRotList(),
    # RandomFlipList(),
    ToTensorList(),  
])

dataset = StegoPairDataset(
    cover_dir=cover_image_folder_path,
    stego_dir=stego_image_folder_path,
    num_samples=100,
    transform=transform,

    bpp=0.4,
    algorithm_name='s-uniward',
    resize_strategy='center_crop',
    W=256,
    H=256,
)

Найдено 7000 stego файлов по заданным параметрам
Загружено 100 пар изображений


# Make DataLoader

In [4]:
# Параметры разделения
val_split = 0.1  
batch_size = 32
random_seed = 42

# Устанавливаем seed для воспроизводимости
torch.manual_seed(random_seed)
np.random.seed(random_seed)

# Вычисляем размеры выборок
dataset_size = len(dataset)
val_size = int(val_split * dataset_size)
train_size = dataset_size - val_size

print(f"Общий размер датасета: {dataset_size}")
print(f"Тренировочная выборка: {train_size}")
print(f"Валидационная выборка: {val_size}")

# Разделяем датасет
train_dataset, val_dataset = random_split(
    dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(random_seed)
)

# Создаем DataLoader'ы для каждой выборки
train_loader = StegoPairDataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # перемешиваем только тренировочные данные
    num_workers=4,
    pin_memory=False,
    drop_last=False
)

val_loader = StegoPairDataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,  # не перемешиваем валидационные данные
    num_workers=4,
    pin_memory=False,
    drop_last=False
)

print(f"Тренировочных батчей: {len(train_loader)}")
print(f"Валидационных батчей: {len(val_loader)}")

Общий размер датасета: 100
Тренировочная выборка: 90
Валидационная выборка: 10
Тренировочных батчей: 6
Валидационных батчей: 1


# Train

In [12]:
import random

def preprocess_data(images, labels, random_crop=True):
    # images of shape: NxCxHxW
    if images.ndim == 5:  # 1xNxCxHxW
        images = images.squeeze(0)
        labels = labels.squeeze(0)
    h, w = images.shape[-2:]

    if random_crop:
        ch = random.randint(h * 3 // 4, h)  # h // 2      #256
        cw = random.randint(w * 3 // 4, w)  # square ch   #256

        h0 = random.randint(0, h - ch)  # 128
        w0 = random.randint(0, w - cw)  # 128
    else:
        ch, cw, h0, w0 = h, w, 0, 0

     
    cw = cw & ~1
    inputs = [
        images[..., h0:h0 + ch, w0:w0 + cw // 2],
        images[..., h0:h0 + ch, w0 + cw // 2:w0 + cw]
    ]

    return inputs, labels

In [ ]:
# Params:
alpha = 0.1
margin = 1
num_epochs = 20
learning_rate = 1e-3
eps = 1e-8
weight_decay = 1e-4


torch.manual_seed(random_seed)
np.random.seed(random_seed)

net = SiaStegNet()
net.train()

criterion_1 = nn.CrossEntropyLoss()
criterion_2 = ContrastiveLoss(margin=margin)
optimizer = Adamax(net.parameters(), lr=learning_rate, eps=eps, weight_decay=weight_decay)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 'max', factor=0.3,
    patience=5, 
    min_lr=0,
    eps=eps
)

# Списки для хранения метрик
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
batch_losses = []
batch_accuracies = []

# Функция валидации
def validate(net: nn.Module, data_loader):
    net.eval()
    running_loss = 0.0
    running_acc = 0.0
    
    with torch.no_grad():
        for batch_data in data_loader:
            images = batch_data['images']
            labels = batch_data['labels']
            inputs, labels = preprocess_data(images, labels)

            optimizer.zero_grad()
            outputs, feats_0, feats_1 = net(*inputs)

            predictions = outputs.argmax(dim=1)
            loss = criterion_1(outputs, labels) + alpha * criterion_2(feats_0, feats_1, labels)
            accuracy = (predictions == labels).float().mean().item()

            running_loss += loss.item()
            running_acc += accuracy
    
    val_loss = running_loss / len(data_loader)
    val_acc = running_acc / len(data_loader)
    
    return val_loss, val_acc

def train(net: nn.Module, data_loader):
    net.train()
    running_loss = 0.0
    running_acc = 0.0
    
    progress_bar = tqdm(data_loader, desc=f'Обучение')    
    for batch_idx, batch_data in enumerate(progress_bar):
        images = batch_data['images']
        labels = batch_data['labels']
        inputs, labels = preprocess_data(images, labels)
        
        optimizer.zero_grad()
        outputs, feats_0, feats_1 = net(*inputs)
        
        predictions = outputs.argmax(dim=1)
        loss = criterion_1(outputs, labels) + alpha * criterion_2(feats_0, feats_1, labels)
        accuracy = (predictions == labels).float().mean().item()

        running_loss += loss.item()
        running_acc += accuracy
        
        # Изменение весов:
        loss.backward()
        optimizer.step()
        
        # Обновляем прогресс-бар
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{accuracy:.4f}',
            'avg_loss': f'{running_loss/(batch_idx+1):.4f}',
            'avg_acc': f'{running_acc/(batch_idx+1):.4f}',
        })
    
    train_loss = running_loss / len(data_loader)
    train_acc = running_acc / len(data_loader)

    return train_loss, train_acc


for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Эпоха {epoch+1}/{num_epochs}")
    print('='*50)
    
    # ===== ТРЕНИРОВКА =====
    train_loss, train_acc = train(net, train_loader)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # ===== ВАЛИДАЦИЯ =====
    val_loss, val_acc = validate(net, val_loader)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"LR = {current_lr:.6f}")
    
    # Вывод результатов эпохи
    print(f"\n📊 Результаты эпохи {epoch+1}:")
    print(f"  Тренировка - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Валидация  - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    # ===== СОХРАНЕНИЕ ЧЕКПОИНТА =====
    is_best = val_acc > best_val_acc
    if is_best: best_val_acc = val_acc
    
    checkpoint_state = {
        'epoch': epoch + 1,
        'state_dict': net.state_dict(),
        'optimizer': optimizer.state_dict(),

        'train_loss': train_loss,
        'train_acc':  train_acc,
        'val_loss':   val_loss,
        'val_acc':    val_acc,
        'best_val_acc': best_val_acc,
    }
    
    save_checkpoint(checkpoint_state, is_best, f'checkpoints/checkpoint_epoch_{epoch+1}.pth.tar')

print("\n" + "="*50)
print("Обучение завершено!")
print("="*50)
print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [7]:
current_lr = optimizer.param_groups[0]['lr']
print(f"LR = {current_lr:.6f}")

LR = 0.001000
